# DMOSpeech 2: Unified Master Evaluation Suite & Live GPU Retest
## Full End-to-End Experimental Protocol: Re-test Synthesis & Tests 1 through 18

**Author:** Rhenel Jhon Sajol (Master's Thesis 2026)  
**Dataset:** Seed-TTS test-en (Speaker-disjoint: Dev 100 / Calibration 200 / Test 200)  
**Conditions:** Clean + 15 dB White Gaussian Noise (AWGN) + Multi-level SNR Sweep (10, 18, 20, 25, 30 dB) + DEMAND Babble Noise  
**Candidate Steps:** $\mathcal{K} = \{4, 5, 6, 8\}$ (plus Steps 3 and 7 on Dev split for headroom analysis)  
**Statistical Framework:** Learn-Then-Test (LTT) Distribution-Free Risk Control $(\alpha = 0.05, \delta = 0.10)$  

---

### Complete 18-Test Architecture Matrix

| Group | # | Test Title | Core Purpose / Metric Target |
| :--- | :--- | :--- | :--- |
| **A. Trust & Measurement** | 1 | **Seed-Noise Floor** | Step 8 ($s_0$) vs Step 8 ($s_1$), and Steps 4/5/6 under both seeds on Test ($N=200$) |
| | 2 | **WER Outlier Diagnostic Audit** | Audits hallucination repetition loops, raw vs. clipped@1.0 vs. median WER |
| | 3 | **Failure-Definition Sensitivity** | $3 \times 3$ sweep of WER floor ($0.10, 0.125, 0.20$) and SIM margin ($0.01, 0.02, 0.03$) |
| **B. Drift Information Content** | 4 | **Spearman Rank Correlation** | Continuous degradation correlation: $d_4, d_5, d_6$ vs $\Delta$SIM and $\Delta$WER on clean speech |
| | 5 | **Drift's Added Value (Hierarchical OLS)** | Regress degradation on $[\hat{S}, \text{dur}, \text{len}]$, then add $d_i$ (measure $\Delta R^2$ with bootstrap CI) |
| | 6 | **ROC-AUC for Worst Degradations** | AUC for worst 10% & 20% degradations, and drift as independent noise detector |
| | 7 | **Alternative Drift Variants on Dev** | Compares relative L2, cosine, L1, $\|v\|$, distance to step-8, and 3-step slope |
| | 8 | **Oracle Headroom Analysis** | Theoretical minimum achievable NFE at $\le \alpha$ risk vs. Step 5 and proposed |
| **C. Certified Policy Comparison** | 9 | **LTT Certification & Test Evaluation** | Calibration ($N=200$) $\to$ Held-Out Test ($N=200$) across 6 certified policies |
| | 10 | **Paired Exact McNemar Tests** | Discordant pairs $(b, c)$ with exact binomial $p$-values (primary vs exploratory) |
| | 11 | **NFE-vs-Risk Pareto Curves** | 2D sweep of $(\theta, \lambda)$ operating points with speaker bootstrap CIs |
| **D. Acoustic Generalization** | 12 | **SNR Sweep Transition Audit** | 10, 15, 18, 20, 25, 30 dB sweep across dev & test speakers |
| | 13 | **$\hat{S}$ Estimator Empirical Validation** | $\hat{S}$ vs true SNR bias/RMSE, and clean speech margin of safety above 20 dB gate |
| | 14 | **100 Random Re-splits Stability** | Stability of certified parameter and coverage probability ($P(\text{Risk} \le 0.05)$) |
| | 15 | **DEMAND Real Babble Noise Sweep** | Non-stationary cafeteria babble noise tolerance at 15 dB and 20 dB |
| **E. Quality, Speed & Perceptual** | 16 | **Two One-Sided Tests (TOST)** | Positive equivalence across WER ($\pm 0.02$), SIM ($\pm 0.02$), UTMOS ($\pm 0.10$) |
| | 17 | **Wall Latency Benchmarking** | Batch size 1, warm-up discarded, 5+ timed repeats, median and IQR |
| | 18 | **Subjective Listening Test Packager** | 50 test speaker audio pairs packaged with manifest for blind ABX / MOS testing |
| **F. Drift Justification** | P12 | **SNR Gate vs. Drift Gate under New Corruptions** | Reverb, babble, DEMAND, telephone, borderline Gaussian on 166 unused speakers; preregistered McNemar + Holm |
| | P13 | **Improved Drift for Noisy Prompts** | Richer zero-NFE signals (per-frame, per-band, CFG gap); explore on dev, confirm once on calib + test with LTT |

**All generated outputs are automatically packaged into `DMOSpeech2_Comprehensive_Test_Results.zip` and downloaded to your machine!**

### Phase 1: Hardware & Compute Environment Verification

In [ ]:
!nvidia-smi
import sys, platform
print(f'Python Version : {sys.version}')
print(f'Operating System: {platform.platform()}')

### Phase 2: Clone Repository & Install Complete Dependencies

In [ ]:
import os, sys, subprocess

# Auto-clone repository if running in a fresh Google Colab instance
if not os.path.exists('Handover Scheduling') and not os.path.exists('/content/Shonget-DMOSpeech2'):
    print('[Setup] Cloning repository into /content/Shonget-DMOSpeech2...')
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
    %cd /content/Shonget-DMOSpeech2
    os.chdir('/content/Shonget-DMOSpeech2')
elif os.path.exists('/content/Shonget-DMOSpeech2'):
    %cd /content/Shonget-DMOSpeech2
    os.chdir('/content/Shonget-DMOSpeech2')
    !git checkout main
    !git pull origin main

print('[Setup] Installing full dependencies...')
%pip install -q pypinyin cached_path torchdiffeq x-transformers ema-pytorch vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub zhon matplotlib
print('[Setup] Environment ready!')

### Phase 3: Fast Download Checkpoints & Seed-TTS Audio Prompts

In [ ]:
import os
!mkdir -p ckpts
if not os.path.exists('ckpts/model_85000.pt'):
    print('[Download] Downloading speech model checkpoint model_85000.pt (4.0 GB)...')
    !wget -c -q https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt

if not os.path.exists('ckpts/model_1500.pt'):
    print('[Download] Downloading duration predictor checkpoint model_1500.pt (270 MB)...')
    !wget -c -q https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print('[Download] Downloading Seed-TTS evaluation dataset audio binaries via huggingface_hub...')
    !rm -rf data/seed-tts-eval
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir='data/seed-tts-eval')
print('[Download] Models and evaluation audio verified!')

### Phase 4: Live GPU Synthesis Benchmark & Retest (Previous Test Re-run)
Runs the live neural ODE teacher integration and student consistency completion across candidate steps {4, 5, 6, 8} on CUDA GPU.
- If running live GPU synthesis, it populates `gpu_grid_all1000.csv`.
- If pre-computed grid datasets exist, it seamlessly verifies them.

In [ ]:
# Live GPU benchmark re-test
import os, torch, pandas as pd

gpu_csv = 'Handover Scheduling/colab/results/gpu_grid_all1000.csv'
if os.path.exists(gpu_csv):
    df_existing = pd.read_csv(gpu_csv)
    print(f'[Dataset Verified] Existing GPU grid found: {len(df_existing)} evaluations across {df_existing.speaker_id.nunique()} speakers!')
    print('To run a fresh live GPU retest on custom speakers, execute: !python "Handover Scheduling/colab/run_gpu_expanded_benchmark.py" --device cuda --scope all1000')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'[Live GPU Retest] Launching live synthesis benchmark on device: {device}...')
    !python "Handover Scheduling/colab/run_gpu_expanded_benchmark.py" --device {device} --scope all1000


### Phase 5: Preregistration Freezing & Split SHA-256 Checksum Verification
Freezes $\alpha=0.05, \delta=0.10$, Definition 3.1 failure constants ($W_0=0.125, \Delta_{\text{SIM}}=0.02, \Delta_{\text{WER}}=0.01$), grids, and manifest SHA-256 checksums BEFORE running evaluation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('Handover Scheduling/colab'))
from run_comprehensive_evaluation_suite import freeze_preregistration_config, export_rich_per_case_log

out_dir = 'Handover Scheduling/colab/results/master_evaluation_suite'
os.makedirs(out_dir, exist_ok=True)

config = freeze_preregistration_config(out_dir)
df_log = export_rich_per_case_log(out_dir)
print(f'Master per-case log table compiled: {len(df_log)} condition pairs logged.')
display(df_log.head(3).T)

### Phase 6: Group A — Can the Results Be Trusted? (Measurement Integrity)
- **Test 1:** Seed-Noise Floor Benchmark (Step 8 $s_0$ vs. $s_1$, and candidate steps under both seeds)
- **Test 2:** WER Outlier Diagnostic Audit (hypothesis strings, hallucination loop diagnosis, clipped vs raw WER)
- **Test 3:** Failure-Definition Sensitivity Sweep ($3 \times 3$ grid of WER ceiling and SIM margin)

In [ ]:
from run_comprehensive_evaluation_suite import (
    run_test_01_seed_noise_floor,
    run_test_02_wer_outlier_audit,
    run_test_03_failure_sensitivity
)

t1 = run_test_01_seed_noise_floor(out_dir)
t2 = run_test_02_wer_outlier_audit(out_dir)
t3 = run_test_03_failure_sensitivity(out_dir)

### Phase 7: Group B — Does Drift Carry Information? (Physical Probing)
- **Test 4:** Spearman rank correlation of $d_4, d_5, d_6$ vs continuous degradation ($\Delta\text{SIM}, \Delta\text{WER}$) on clean speech
- **Test 5:** Drift's added value (hierarchical nested OLS regression: Baseline vs Augmented, $\Delta R^2$ with 1,000 bootstrap CI)
- **Test 6:** ROC-AUC for worst 10% & 20% degradations, and drift AUC for noisy vs clean
- **Test 7:** Alternative drift variants compared on dev split (cosine, L1, $\|v\|$, distance to 8, slope)
- **Test 8:** Oracle headroom analysis (theoretical minimum NFE at $\le \alpha$ risk)

In [ ]:
from run_comprehensive_evaluation_suite import (
    run_test_04_spearman_drift,
    run_test_05_drift_added_value,
    run_test_06_auc_worst_degradation,
    run_test_07_dev_drift_variants,
    run_test_08_oracle_headroom
)

t4 = run_test_04_spearman_drift(out_dir)
t5 = run_test_05_drift_added_value(out_dir)
t6 = run_test_06_auc_worst_degradation(out_dir)
t7 = run_test_07_dev_drift_variants(out_dir)
t8 = run_test_08_oracle_headroom(out_dir)

### Phase 8: Group C — Certified Policy Comparison Under LTT (Tests 9, 10, 11)
- **Test 9:** Learn-Then-Test certification on calibration ($N=200$) & evaluation on held-out test ($N=200$) across 6 policies with exact Clopper-Pearson binomial CIs
- **Test 10:** Paired exact McNemar tests (discordant counts $b, c$; exact two-tailed binomial $p$-values; pre-specified primary vs exploratory)
- **Test 11:** NFE-vs-Risk Pareto curve data & frontier over $(\theta, \lambda)$ sweeps

In [ ]:
from run_comprehensive_evaluation_suite import (
    run_test_09_ltt_policy_comparison,
    run_test_10_paired_mcnemar,
    run_test_11_pareto_frontiers
)

t9 = run_test_09_ltt_policy_comparison(out_dir)
t10 = run_test_10_paired_mcnemar(out_dir)
t11 = run_test_11_pareto_frontiers(out_dir)

### Phase 9: Group D — Acoustic Generalization & Robustness (Tests 12, 13, 14, 15)
- **Test 12:** SNR sweep transition audit (10, 15, 18, 20, 25, 30 dB)
- **Test 13:** $\hat{S}$ estimator validation (bias, RMSE, clean speech margin of safety)
- **Test 14:** 100 random re-splits stability audit (coverage probability $P(\text{Risk} \le 0.05)$)
- **Test 15:** DEMAND real cafeteria babble noise sweep (15 dB & 20 dB)

In [ ]:
from run_comprehensive_evaluation_suite import (
    run_test_12_snr_sweep,
    run_test_13_snr_estimator_validation,
    run_test_14_resplit_stability,
    run_test_15_demand_real_noise
)

t12 = run_test_12_snr_sweep(out_dir)
t13 = run_test_13_snr_estimator_validation(out_dir)
t14 = run_test_14_resplit_stability(out_dir)
t15 = run_test_15_demand_real_noise(out_dir)

### Phase 10: Group E — Quality, Speed & Perceptual Testing (Tests 16, 17, 18)
- **Test 16:** Two One-Sided Tests (TOST) on clipped WER, SIM, UTMOS on clean-only and overall
- **Test 17:** Latency benchmark (batch size 1, warm-up discarded, median, IQR, speedup)
- **Test 18:** Subjective listening test audio packager (50 test speaker audio pairs with manifest)

In [ ]:
from run_comprehensive_evaluation_suite import (
    run_test_16_tost_equivalence,
    run_test_17_latency_benchmark,
    run_test_18_listening_test_packager,
    generate_all_publication_figures,
    generate_comprehensive_markdown_report
)

t16 = run_test_16_tost_equivalence(out_dir)
t17 = run_test_17_latency_benchmark(out_dir)
t18 = run_test_18_listening_test_packager(out_dir)

generate_all_publication_figures(out_dir)
generate_comprehensive_markdown_report({}, out_dir)

### Phase 11: Publication Visualizations & Master Summary Report Display
Displays Figure 1 (Pareto Operating Curve), Figure 2 (TOST Equivalence 90% Confidence Intervals), and the complete executive summary report inline for immediate audit.

In [ ]:
from IPython.display import Image, display, Markdown

fig1 = os.path.join(out_dir, 'figures', 'fig_01_pareto_curve.png')
fig2 = os.path.join(out_dir, 'figures', 'fig_02_tost_equivalence.png')

if os.path.exists(fig1):
    print('=== Figure 1: Pareto Operating Frontier (NFE vs. Risk) ===')
    display(Image(fig1))

if os.path.exists(fig2):
    print('\n=== Figure 2: TOST Equivalence 90% Confidence Intervals ===')
    display(Image(fig2))

rep_path = os.path.join(out_dir, 'CHAPTER3_MASTER_EVALUATION_REPORT.md')
if os.path.exists(rep_path):
    with open(rep_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

### Phase 12: Drift Justification Benchmark (SNR Gate vs. Drift Gate under New Corruptions)
The Chapter 3 data (clean + 15 dB Gaussian) showed drift is **redundant** with the SNR gate: SNR-only (2.0% risk, 18.03 NFE) beat SNR+drift (2.5%, 18.15 NFE). This phase tests whether drift catches prompt degradations that the energy-percentile SNR estimator **cannot see**.

| Condition | What it is | Why the SNR gate may miss it |
| :--- | :--- | :--- |
| `reverb03/06/09` | Synthetic room impulse response, RT60 0.3 / 0.6 / 0.9 s | Reverb is not additive noise, so the SNR estimate stays high |
| `babble15/20` | Mixture of 6 other speakers' prompts at 15 / 20 dB | Non-stationary speech-shaped noise |
| `demand15/20` | Real DEMAND cafeteria noise (PCAFETER) at 15 / 20 dB | Non-stationary real noise (optional download) |
| `phone` | 8 kHz telephone band (300-3400 Hz) + 8-bit mu-law | High SNR but band-limited, quantized prompt |
| `gauss18/20/22/25` | Borderline white Gaussian noise | Right at the 20 dB gate, where the estimator bias matters |

**Frozen before any synthesis** (written to `drift_justification_preregistration.json`):
- Policies (early handover at Step 5, else Step 8): `snr_only` ($\hat{S} \ge 20$ dB), `drift_only` ($d_4 > 0.30$), `snr_and_drift` (both).
- Prerequisite: a condition counts only if Step-5 failure is significantly above the 2.5% seed-noise floor.
- Primary hypothesis: on at least one eligible condition, `snr_and_drift` has fewer failures than `snr_only` (one-sided exact McNemar, Holm-corrected).
- Speakers: the **166 Seed-TTS speakers not used** in Dev/Calib/Test, so no condition or threshold is tuned on them.

The run resumes from its CSV checkpoint if the Colab session disconnects; just re-run the cell.

In [ ]:
# Optional: download real DEMAND cafeteria noise (PCAFETER, 16 kHz). Skip if it fails;
# demand15/demand20 are then SKIPPED by the runner (no substitute numbers are generated).
import os
demand_wav = 'data/demand/PCAFETER/ch01.wav'
if not os.path.exists(demand_wav):
    os.makedirs('data/demand', exist_ok=True)
    !wget -q -c "https://zenodo.org/records/1227121/files/PCAFETER_16k.zip?download=1" -O data/demand/PCAFETER_16k.zip
    !unzip -q -o data/demand/PCAFETER_16k.zip -d data/demand
print('[DEMAND] available' if os.path.exists(demand_wav) else '[DEMAND] not available -> demand conditions will be skipped')

In [ ]:
# Choose what to run:
#   'smoke'  : 5 speakers x quick conditions (~minutes) -> checks the pipeline end to end
#   'quick'  : 166 fresh speakers x [clean, reverb06, phone]  (cheapest real answer)
#   'reverb' : 166 fresh speakers x [clean, reverb03, reverb06, reverb09]
#   'all'    : 166 fresh speakers x all 13 conditions (several hours on a T4)
# You can run 'quick' first and 'all' later: finished pairs are reused from the checkpoint CSV.
RUN_MODE = 'quick'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner = '"Handover Scheduling/colab/run_drift_justification_benchmark.py"'
drift_out = 'Handover Scheduling/colab/results/drift_justification'

if RUN_MODE == 'smoke':
    # Separate output dir so smoke rows never mix with the real study
    !python {runner} --device {device} --conditions quick --n_speakers 5 --output_dir "{drift_out}_smoke"
else:
    !python {runner} --device {device} --conditions {RUN_MODE} --output_dir "{drift_out}"

In [ ]:
# Display the computed drift justification report (every number comes from the grid CSV)
import os, json
from IPython.display import Markdown, display

drift_out = 'Handover Scheduling/colab/results/drift_justification'
rep = os.path.join(drift_out, 'DRIFT_JUSTIFICATION_REPORT.md')
if os.path.exists(rep):
    with open(rep, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f'No report yet at {rep}. Run the previous cell with RUN_MODE != "smoke".')

### Phase 13: Improved Drift (Richer Zero-NFE Stabilization Signals for Noisy Prompts)
Phase 12 showed global drift $d_i$ adds nothing over the SNR gate. On clean prompts nothing can help (Step-5 failures sit at the 2.5% seed-noise floor), but **about half of noisy prompts are safe at Step 5** and the SNR gate sends all of them to Step 8. This phase tests whether a richer stabilization signal can rescue them:

$\hat{S} \ge \theta \Rightarrow$ Step 5; $\quad \hat{S} < \theta$ and drift risk score $\le \lambda \Rightarrow$ Step 5; $\quad$ otherwise Step 8.

Features (all from steps $\le 5$, all **zero extra forward passes**): global drift, per-frame drift (mean / p90 / max), per-band drift (low / mid / high mel), cosine change, CFG disagreement $\|v_c - v_u\|$, velocity norm, endpoint spread.

**Flow:**
1. **Extract** (GPU, teacher only, no Whisper/WavLM/UTMOS): all 1,000 pairs. Failure labels are reused from `gpu_grid_all1000.csv`; the run must reproduce the stored $d_4, d_5, d_6$.
2. **Explore** on the 100 development speakers only (repeat as often as needed; calibration and test stay unseen).
3. **Confirm once** on calibration + test: Holm-corrected AUC, LTT certification, held-out risk / NFE vs. SNR→4, SNR→5 and the original policy, plus a matched-random control.

In [ ]:
# 13a. Smoke test: 10 teacher-only passes (a few minutes). Writes to a separate folder.
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
feat_runner = '"Handover Scheduling/colab/run_drift_features_benchmark.py"'
feat_out = 'Handover Scheduling/colab/results/drift_features'
!python {feat_runner} --device {device} --n_pairs 10 --output_dir "{feat_out}_smoke"

In [ ]:
# 13b. Full extraction (1,000 teacher-only passes, resumable) + EXPLORE on development speakers only.
# Calibration and test speakers are NOT scored here.
!python {feat_runner} --device {device} --output_dir "{feat_out}" --explore_only

# Download the feature CSV so the exploration can also be repeated offline (no GPU needed)
try:
    from google.colab import files
    files.download(f'{feat_out}/drift_features_all1000.csv')
except Exception:
    print(f'Download manually: {feat_out}/drift_features_all1000.csv')

In [ ]:
# 13c. CONFIRM ONCE on calibration + test (run only after the exploration is final).
!python {feat_runner} --output_dir "{feat_out}" --analyze_only

import os
from IPython.display import Markdown, display
rep = os.path.join(feat_out, 'DRIFT_FEATURES_REPORT.md')
if os.path.exists(rep):
    with open(rep, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

### Phase 14: Master Artifact ZIP Archive & Automatic Colab Download
Bundles all CSV logs, JSON results for all 18 tests, sensitivity grids, Pareto curves, figures, report, and listening test manifests, and the Phase 12-13 drift results into a single archive and downloads it.

In [ ]:
import os, zipfile

zip_path = 'Handover Scheduling/colab/DMOSpeech2_Comprehensive_Test_Results.zip'
print(f'[Packaging] Packing all evaluation outputs into {zip_path}...')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(out_dir):
        for fname in files:
            full_path = os.path.join(root, fname)
            rel_path = os.path.relpath(full_path, out_dir)
            zf.write(full_path, rel_path)

    # Phase 12-13 drift results (kept in their own results folder)
    for sub in ['drift_justification', 'drift_features']:
        drift_out = f'Handover Scheduling/colab/results/{sub}'
        if os.path.exists(drift_out):
            for root, dirs, files in os.walk(drift_out):
                for fname in files:
                    full_path = os.path.join(root, fname)
                    zf.write(full_path, os.path.join(sub, os.path.relpath(full_path, drift_out)))

print(f'[Packaging Complete] Successfully created bundle: {zip_path} ({os.path.getsize(zip_path):,} bytes)!')

# Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download(zip_path)
    print('[Download] Automatic browser download triggered in Google Colab!')
except Exception:
    print(f'[Download] Download manually from left sidebar: {zip_path}')